# Fase 1 · Carga de datos — Resultados Saber 11 (ICFES)

**Proyecto:** Alerta temprana de bajo desempeño en la prueba Saber 11
**Autor:** Juan Carlos Andrade Unigarro
**Programa:** Machine Learning and Data Science (MLDS) · Módulo 6 · Universidad Nacional de Colombia

Este notebook ejecuta y documenta la carga de datos del proyecto. Usa las funciones del paquete
`saber11` (`src/saber11/database/carga.py`), las mismas que invoca el script
`scripts/data_acquisition/main.py`, de modo que la carga se puede reproducir desde la línea de comandos
o desde aquí.

Qué hace:

1. Consulta la fuente (datos.gov.co) y cuenta los registros disponibles.
2. Descarga el recorte de datos del proyecto y lo guarda en `data/raw`.
3. Carga los datos con sus tipos y elimina las filas duplicadas que trae la fuente.
4. Verifica que la carga quedó completa y coherente.

La descripción de cada variable está en `docs/data/data_dictionary.md` y el detalle de la fuente en
`docs/data/data_definition.md`.

## 1. Preparación del entorno

La celda ubica la raíz del repositorio. Si el notebook se abre fuera de él (por ejemplo en Google Colab),
clona el repositorio para tener disponible el paquete `saber11`. Solo se necesitan `pandas` y `requests`.

In [1]:
import logging
import os
import subprocess
import sys
from pathlib import Path

URL_REPOSITORIO = "https://github.com/Yosoyepa/mlds6-saber11.git"


def ubicar_raiz() -> Path:
    """Devuelve la raíz del repositorio; lo clona si el notebook se ejecuta fuera de él."""
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "pyproject.toml").exists() and (carpeta / "src" / "saber11").exists():
            return carpeta
    destino = Path.cwd() / "mlds6-saber11"
    if not destino.exists():
        subprocess.run(["git", "clone", "--quiet", URL_REPOSITORIO, str(destino)], check=True)
    return destino


RAIZ = ubicar_raiz()
os.chdir(RAIZ)  # a partir de aquí todas las rutas son relativas a la raíz del repositorio
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import requests

from saber11.database import carga

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
pd.set_option("display.max_columns", 60)

print("Python  ", sys.version.split()[0])
print("pandas  ", pd.__version__)
print("requests", requests.__version__)

Python   3.12.10
pandas   3.1.1
requests 2.34.2


## 2. Fuente y recorte de datos

| | |
|---|---|
| **Conjunto de datos** | Resultados únicos Saber 11 |
| **Publicador** | Instituto Colombiano para la Evaluación de la Educación (ICFES) |
| **Portal** | Datos Abiertos de Colombia — <https://www.datos.gov.co/d/kgxf-xxbe> |
| **Acceso** | API SODA (Socrata), pública y sin credenciales |
| **Licencia** | Creative Commons Attribution-ShareAlike 4.0 |
| **Cobertura** | 7,1 millones de registros, periodos 2010-1 a 2022-4, 51 columnas |

El proyecto trabaja con el periodo **2022-4** (aplicación de calendario A de 2022, la más reciente del
recurso) y, dentro de él, con una **muestra sistemática de 1 de cada 10 examinados**: los registros cuyo
`estu_consecutivo` termina en el dígito 7. Como el consecutivo se asigna de forma secuencial, el filtro es
reproducible sin depender de una semilla aleatoria y se aplica en el servidor, así que solo se descarga lo
necesario.

In [2]:
PERIODO = carga.PERIODO_POR_DEFECTO      # "20224"
DIGITOS = carga.DIGITOS_POR_DEFECTO      # ("7",)
FORZAR_DESCARGA = False                  # True para descargar de nuevo aunque el archivo exista

print("Filtro enviado a la API:", carga.construir_filtro(PERIODO, DIGITOS))

filas_periodo = carga.contar_registros(PERIODO, digitos=None)
filas_muestra = carga.contar_registros(PERIODO, DIGITOS)
print(f"Filas del periodo {PERIODO} en la fuente : {filas_periodo:,}")
print(f"Filas del recorte del proyecto          : {filas_muestra:,} ({filas_muestra / filas_periodo:.1%})")

Filtro enviado a la API: periodo = '20224' AND (estu_consecutivo like '%7')


Filas del periodo 20224 en la fuente : 1,065,888
Filas del recorte del proyecto          : 106,850 (10.0%)


## 3. Descarga y almacenamiento de los datos crudos

La descarga se hace por páginas de 50.000 filas ordenadas por el identificador interno de fila, con
reintentos ante fallos de red. Todas las columnas se leen como texto para no alterar los códigos con ceros
a la izquierda (DANE, departamento, municipio). Al terminar se comprueba que el número de filas recibidas
coincida con el conteo que reporta la API.

Los datos crudos se guardan sin modificar en `data/raw/` (carpeta excluida del control de versiones), junto
con un archivo `.meta.json` que registra la consulta, la fecha de descarga y la huella SHA-256 del archivo.

In [3]:
ruta = carga.ruta_por_defecto("data/raw", PERIODO, DIGITOS)

if FORZAR_DESCARGA or not ruta.exists():
    crudo = carga.descargar_saber11(PERIODO, DIGITOS)
    carga.guardar_crudo(crudo, ruta, carga.construir_filtro(PERIODO, DIGITOS))
else:
    print(f"Se reutiliza el archivo local {ruta.as_posix()}")

crudo = carga.leer_crudo(ruta)
print()
print(f"Archivo crudo: {ruta.as_posix()} ({ruta.stat().st_size / 1e6:.1f} MB)")
print(f"Dimensiones  : {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")
print()
print("Metadatos de la descarga:")
print(ruta.with_suffix(".meta.json").read_text(encoding="utf-8"))

01:10:00 | INFO | La fuente reporta 106850 filas para el filtro: periodo = '20224' AND (estu_consecutivo like '%7')


01:11:11 | INFO | Descargadas 50000 de 106850 filas.


01:11:18 | INFO | Descargadas 100000 de 106850 filas.


01:11:21 | INFO | Descargadas 106850 de 106850 filas.


01:11:26 | INFO | Datos crudos guardados en data\raw\saber11_20224_d7.csv (106850 filas).



Archivo crudo: data/raw/saber11_20224_d7.csv (45.5 MB)
Dimensiones  : 106,850 filas x 51 columnas

Metadatos de la descarga:
{
  "fuente": "Resultados únicos Saber 11 - ICFES",
  "ficha": "https://www.datos.gov.co/d/kgxf-xxbe",
  "api": "https://www.datos.gov.co/resource/kgxf-xxbe.csv",
  "licencia": "Creative Commons Attribution-ShareAlike 4.0 International",
  "filtro_soql": "periodo = '20224' AND (estu_consecutivo like '%7')",
  "fecha_descarga_utc": "2026-11-26T06:11:26+00:00",
  "filas": 106850,
  "columnas": 51,
  "sha256": "2581985d3ca86af07a7e0f02bc2fbdd1bbe86c16f79960084685b350e781a588"
}


## 4. Carga para análisis

`cargar_saber11` lee el archivo crudo y aplica dos operaciones propias de la carga:

- **Elimina las filas duplicadas exactas.** El recurso publica dos veces cada registro del periodo 2022-4
  (lo mismo ocurre con 2019-4). Son copias idénticas en las 51 columnas, así que quitarlas no descarta
  información.
- **Asigna los tipos de dato.** Los puntajes pasan a numéricos y la fecha de nacimiento a fecha. Los códigos
  se conservan como texto porque son identificadores.

La limpieza de fondo (imputación, recodificación, valores atípicos) queda para la fase de preprocesamiento.

In [4]:
datos = carga.cargar_saber11(ruta)

print()
print(f"Filas crudas                : {len(crudo):,}")
print(f"Filas duplicadas eliminadas : {len(crudo) - len(datos):,}")
print(f"Filas cargadas (examinados) : {len(datos):,}")
print(f"Columnas                    : {datos.shape[1]}")
datos.head()

01:11:29 | INFO | Se eliminaron 53443 filas duplicadas exactas de 106850.



Filas crudas                : 106,850
Filas duplicadas eliminadas : 53,443
Filas cargadas (examinados) : 53,407
Columnas                    : 51


,periodo,estu_tipodocumento,estu_consecutivo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_cod_dane_establecimiento,cole_cod_dane_sede,cole_cod_depto_ubicacion,cole_cod_mcpio_ubicacion,cole_codigo_icfes,cole_depto_ubicacion,cole_genero,cole_jornada,cole_mcpio_ubicacion,cole_naturaleza,cole_nombre_establecimiento,cole_nombre_sede,cole_sede_principal,estu_cod_depto_presentacion,estu_cod_mcpio_presentacion,estu_cod_reside_depto,estu_cod_reside_mcpio,estu_depto_presentacion,estu_depto_reside,estu_estadoinvestigacion,estu_estudiante,estu_fechanacimiento,estu_genero,estu_mcpio_presentacion,estu_mcpio_reside,estu_nacionalidad,estu_pais_reside,estu_privado_libertad,fami_cuartoshogar,fami_educacionmadre,fami_educacionpadre,fami_estratovivienda,fami_personashogar,fami_tieneautomovil,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,desemp_ingles,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global
0,20224,CC,SB11202240478157,URBANO,N,A,ACADÉMICO,163594000389,163594000389,63,63594,739631,QUINDIO,MIXTO,UNICA,QUIMBAYA,OFICIAL,INSTITUCION EDUCATIVA POLICARPA SALAVARRIETA,POLICARPA SALAVARRIETA,S,63,63594,63,63594,QUINDIO,QUINDIO,PUBLICAR,ESTUDIANTE,2004-05-15,M,QUIMBAYA,QUIMBAYA,COLOMBIA,COLOMBIA,N,NaN,NaN,NaN,NaN,3 a 4,Si,Si,NaN,No,A2,59.0,57.0,49.0,38.0,57.0,255.0
1,20224,TI,SB11202240450347,URBANO,NaN,A,TÉCNICO,376147000961,376147000961,76,76147,125112,VALLE,MIXTO,MAÑANA,CARTAGO,NO OFICIAL,COL. DIOCESANO JUAN XXIII,COL. DIOCESANO JUAN XXIII,S,76,76147,76,76147,VALLE,VALLE,PUBLICAR,ESTUDIANTE,2005-07-21,M,CARTAGO,CARTAGO,COLOMBIA,COLOMBIA,N,Tres,NaN,NaN,NaN,3 a 4,Si,Si,NaN,Si,A1,48.0,47.0,51.0,53.0,48.0,248.0
2,20224,CC,SB11202240231487,URBANO,N,A,TÉCNICO/ACADÉMICO,111001008389,111001008389,11,11001,020701,BOGOTÁ,MIXTO,TARDE,BOGOTÁ D.C.,OFICIAL,COLEGIO TECNICO DOMINGO FAUSTINO SARMIENTO (IED),INST TEC DIST DOMINGO FAUSTINO SARMIENTO,S,11,11001,11,11001,BOGOTÁ,BOGOTÁ,PUBLICAR,ESTUDIANTE,2002-01-18,M,BOGOTÁ D.C.,BOGOTÁ D.C.,COLOMBIA,COLOMBIA,N,Tres,Primaria completa,Primaria incompleta,Estrato 3,5 a 6,No,Si,Si,Si,A2,59.0,58.0,44.0,58.0,64.0,281.0
3,20224,CC,SB11202240460927,URBANO,N,A,TÉCNICO,173861000721,173861000721,73,73861,741165,TOLIMA,MIXTO,UNICA,VENADILLO,OFICIAL,INSTITUCION EDUCATIVA TECNICA COMERCIAL CAMILA...,I.E.T. COMERCIAL CAMILA MOLANO - SEDE PRINCIPAL,S,73,73861,73,73861,TOLIMA,TOLIMA,PUBLICAR,ESTUDIANTE,2003-09-12,M,VENADILLO,VENADILLO,COLOMBIA,COLOMBIA,N,Dos,Secundaria (Bachillerato) completa,Primaria completa,Estrato 1,3 a 4,No,Si,Si,Si,A-,34.0,42.0,34.0,37.0,43.0,193.0
4,20224,TI,SB11202240403897,URBANO,N,A,TÉCNICO,173001011237,173001011237,73,73001,743781,TOLIMA,MIXTO,UNICA,IBAGUÉ,OFICIAL,INST EDUC MUS AMINA MELENDRO DE PULECIO,AMINA MELENDRO DE PULECIO,S,73,73001,73,73001,TOLIMA,TOLIMA,PUBLICAR,ESTUDIANTE,2005-08-06,F,IBAGUÉ,IBAGUÉ,COLOMBIA,COLOMBIA,N,Cuatro,Técnica o tecnológica incompleta,Secundaria (Bachillerato) completa,Estrato 2,5 a 6,Si,Si,Si,Si,A1,55.0,58.0,52.0,51.0,67.0,284.0


## 5. Verificaciones de la carga

Comprobaciones automáticas: si alguna falla, la celda se detiene con un error.

In [5]:
assert list(datos.columns) == carga.COLUMNAS, "El esquema no coincide con el esperado"
assert datos[carga.LLAVE].is_unique, "Hay examinados repetidos"
assert datos[carga.LLAVE].notna().all(), "Hay examinados sin identificador"
assert set(datos["periodo"]) == {PERIODO}, "Hay registros de otros periodos"
assert datos[carga.LLAVE].str.endswith(tuple(DIGITOS)).all(), "Hay registros fuera de la muestra"
assert datos["punt_global"].between(0, 500).all(), "Puntaje global fuera del rango 0-500"
for columna in carga.COLUMNAS_PUNTAJE[:-1]:
    assert datos[columna].dropna().between(0, 100).all(), f"{columna} fuera del rango 0-100"
assert pd.api.types.is_datetime64_any_dtype(datos["estu_fechanacimiento"]), "La fecha no quedó como fecha"

print("Esquema completo: 51 columnas en el orden de la fuente")
print(f"Llave única     : {datos[carga.LLAVE].nunique():,} consecutivos para {len(datos):,} filas")
print(f"Periodo         : {sorted(datos['periodo'].unique())}")
print("Puntajes        : dentro de los rangos oficiales (0-100 por prueba, 0-500 global)")
print("Todas las verificaciones pasaron.")

Esquema completo: 51 columnas en el orden de la fuente
Llave única     : 53,407 consecutivos para 53,407 filas
Periodo         : ['20224']
Puntajes        : dentro de los rangos oficiales (0-100 por prueba, 0-500 global)
Todas las verificaciones pasaron.


### Tipos, cardinalidad y valores faltantes por columna

Esta tabla es el insumo del diccionario de datos (`docs/data/data_dictionary.md`).

In [6]:
perfil = pd.DataFrame(
    {
        "tipo": datos.dtypes.astype(str),
        "valores_unicos": datos.nunique(),
        "faltantes": datos.isna().sum(),
        "faltantes_%": (datos.isna().mean() * 100).round(2),
    }
)
perfil.index.name = "variable"
with pd.option_context("display.max_rows", 60):
    display(perfil)

,tipo,valores_unicos,faltantes,faltantes_%
variable,,,,
periodo,str,1,0,0.00
estu_tipodocumento,str,10,0,0.00
estu_consecutivo,str,53407,0,0.00
cole_area_ubicacion,str,2,0,0.00
cole_bilingue,str,2,9665,18.10
cole_calendario,str,3,0,0.00
cole_caracter,str,4,2116,3.96
cole_cod_dane_establecimiento,str,9159,0,0.00
cole_cod_dane_sede,str,9726,0,0.00


### Puntajes

`punt_global` es la variable de la que se deriva el objetivo del proyecto (bajo desempeño). Aquí solo se
confirma que se cargó con valores plausibles; su análisis corresponde a la fase de exploración.

In [7]:
datos[carga.COLUMNAS_PUNTAJE].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
punt_ingles,53224.0,49.83,13.14,0.0,39.0,49.0,58.0,100.0
punt_matematicas,53407.0,50.56,12.25,15.0,42.0,51.0,59.0,100.0
punt_sociales_ciudadanas,53407.0,47.73,11.97,0.0,38.0,47.0,57.0,100.0
punt_c_naturales,53407.0,49.06,10.39,0.0,41.0,48.0,56.0,100.0
punt_lectura_critica,53407.0,52.75,10.81,0.0,45.0,53.0,61.0,100.0
punt_global,53407.0,250.04,51.97,88.0,209.0,247.0,287.5,450.0


### Composición de la muestra

Una mirada rápida a tres variables de contexto para confirmar que la muestra conserva la composición
esperada del examen (mayoría de colegios oficiales y urbanos, estratos 1 a 3).

In [8]:
for columna in ["cole_naturaleza", "cole_area_ubicacion", "fami_estratovivienda"]:
    distribucion = (datos[columna].value_counts(normalize=True, dropna=False) * 100).round(1)
    print(distribucion.rename_axis(columna).rename("%").to_string())
    print()

cole_naturaleza
OFICIAL       77.3
NO OFICIAL    22.7

cole_area_ubicacion
URBANO    82.9
RURAL     17.1

fami_estratovivienda
Estrato 2      33.9
Estrato 1      27.2
Estrato 3      20.8
NaN             6.7
Estrato 4       5.2
Sin Estrato     3.8
Estrato 5       1.6
Estrato 6       0.7



## 6. Resultado

- La fuente respondió sin credenciales y la descarga quedó completa: el número de filas recibidas coincide
  con el que reporta la API.
- El recorte del proyecto tiene **106.850 filas crudas**, que corresponden a **53.407 examinados** una vez
  eliminadas las 53.443 copias exactas que publica la fuente para este periodo.
- Los datos cargados tienen **51 columnas**: 44 de texto (categorías, códigos y nombres), 6 puntajes
  numéricos y 1 fecha.
- Los valores faltantes se concentran en `cole_bilingue` (18 %) y en el cuestionario socioeconómico
  (`fami_*`, entre 4 % y 7 %). Los puntajes están completos, salvo inglés (0,3 %).

Los datos quedan disponibles para las fases siguientes mediante una sola llamada:

```python
from saber11.database import obtener_saber11
datos = obtener_saber11()
```